# AI 기본법 적응형 계층 청킹 + Anchor + Parent Retrieval

`split.py`의 조 → 항 → 호 → 목 계층 분할을 바탕으로 다음을 반영한 노트북입니다.

- 800자 이하 조문은 조 단위 검색 청크 1개로 유지
- 긴 조문은 전체 원문을 `parent`로 보존하고 필요한 깊이까지만 분할
- 긴 항·호는 핵심 문장만 담은 `anchor`를 추가
- 장·절 제목은 단독 청크가 아니라 metadata로만 보존
- Qdrant 검색은 `search_enabled=True`인 `child/anchor`만 대상으로 수행
- 검색된 자식의 `parent_point_id`를 이용해 전체 조문을 답변 근거로 확장

기존 컬렉션을 건드리지 않고 `ai_basic_law_adaptive_anchor_v1`을 새로 만듭니다.

In [1]:
from pathlib import Path
from dataclasses import dataclass, field
from collections import Counter
import asyncio
import hashlib
import re
import sys
import uuid
import xml.etree.ElementTree as ET

from dotenv import load_dotenv
from qdrant_client.models import (
    Distance, FieldCondition, Filter, MatchValue,
    PointStruct, VectorParams,
)

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / 'common' / 'ai_model.py').exists()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError('프로젝트 폴더에서 노트북을 실행하세요.')

sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / '.env', override=True)

from common.ai_model import get_embedding_model
from common.qdrant import get_qdrant_client

XML_PATH = PROJECT_ROOT / 'data' / 'ai_basic_law' / 'ai_basic_law.xml'
COLLECTION_NAME = 'ai_basic_law_adaptive_anchor_v1'
MAX_CHARS = 800
OVERLAP = 80
LAW_VERSION = '282791'

if not XML_PATH.exists():
    raise FileNotFoundError(f'XML 파일이 없습니다: {XML_PATH}')

root = ET.parse(XML_PATH).getroot()
print('XML:', XML_PATH)
print('루트 태그:', root.tag)

XML: /home/aiuser/rag_mini_pjt_template/data/ai_basic_law/ai_basic_law.xml
루트 태그: 법령


## 1. 적응형 계층 청킹

구조 경계가 우선이며 글자 수 분할은 구조 분할 후에도 너무 긴 leaf에만 적용합니다. `anchor`는 긴 목록의 정의·의무 도입 문장이 검색에서 사라지는 문제를 막습니다.

In [2]:
LEVELS = {'article': 0, 'paragraph': 1, 'item': 2, 'subitem': 3}
CHAPTER_RE = re.compile(r'^제\d+장')
SECTION_RE = re.compile(r'^제\d+[절관]')

def clean(value):
    return ' '.join((value or '').split())

def direct_text(element, tag):
    return clean(element.findtext(f'./{tag}', ''))

def digits(value):
    found = re.sub(r'\D', '', value or '')
    return found or None

def paragraph_no(value):
    value = (value or '').strip()
    if len(value) == 1 and 0x2460 <= ord(value) <= 0x2473:
        return str(ord(value) - 0x2460 + 1)
    return digits(value)

@dataclass
class Node:
    level: str
    own: str
    no: str | None = None
    branch: str | None = None
    children: list['Node'] = field(default_factory=list)

def build_tree(unit):
    def make_subitems(item):
        return [
            Node('subitem', direct_text(mok, '목내용'),
                 direct_text(mok, '목번호').strip('. ') or None)
            for mok in item.findall('./목')
        ]

    def make_items(parent):
        return [
            Node(
                'item', direct_text(item, '호내용'),
                digits(direct_text(item, '호번호')),
                direct_text(item, '호가지번호') or None,
                make_subitems(item),
            )
            for item in parent.findall('./호')
        ]

    article = Node(
        'article', direct_text(unit, '조문내용'),
        direct_text(unit, '조문번호'),
        direct_text(unit, '조문가지번호') or None,
    )
    for paragraph in unit.findall('./항'):
        article.children.append(Node(
            'paragraph', direct_text(paragraph, '항내용'),
            paragraph_no(direct_text(paragraph, '항번호')),
            children=make_items(paragraph),
        ))
    article.children.extend(make_items(unit))
    return article

def render(node):
    lines = [node.own] if node.own else []
    for child in node.children:
        child_text = render(child)
        if child_text:
            lines.append(child_text)
    return '\n'.join(lines)

def article_label(node):
    return f'제{node.no}조' + (f'의{node.branch}' if node.branch else '')

def location_segment(node):
    if node.level == 'paragraph':
        return f'{node.no}항' if node.no else ''
    if node.level == 'item':
        return f'{node.no}' + (f'의{node.branch}' if node.branch else '') + '호'
    if node.level == 'subitem':
        return f'{node.no}목' if node.no else ''
    return ''

def split_leaf(text, size=MAX_CHARS, overlap=OVERLAP):
    if len(text) <= size:
        return [text]
    step = size - overlap
    return [text[start:start + size] for start in range(0, len(text), step)]

def stable_point_id(chunk_key):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, chunk_key))

def build_chunks(root, max_chars=MAX_CHARS):
    info = root.find('./기본정보')
    law_name = direct_text(info, '법령명_한글')
    law_id = direct_text(info, '법령ID')
    common = {
        'law_name': law_name,
        'law_id': law_id,
        'law_version': LAW_VERSION,
        'promulgation_date': direct_text(info, '공포일자'),
        'enforce_date': direct_text(info, '시행일자'),
        'source': XML_PATH.name,
    }
    chunks = []
    chapter = section = None

    def add(content, search_text, metadata, chunk_key):
        chunks.append({
            'point_id': stable_point_id(chunk_key),
            'chunk_key': chunk_key,
            'content': content,
            'search_text': search_text,
            **common,
            **metadata,
        })

    for unit in root.findall('./조문/조문단위'):
        if direct_text(unit, '조문여부') == '전문':
            heading = direct_text(unit, '조문내용')
            if CHAPTER_RE.match(heading):
                chapter, section = heading, None
            elif SECTION_RE.match(heading):
                section = heading
            continue

        article = build_tree(unit)
        article_name = article_label(article)
        title = direct_text(unit, '조문제목')
        title_label = f'{article_name}({title})' if title else article_name
        full_text = render(article)
        article_key = f'{LAW_VERSION}:{law_id}:{article_name}'
        parent_key = f'{article_key}:parent'
        parent_point_id = stable_point_id(parent_key)
        base = {
            'source_type': 'article', 'chapter': chapter, 'section': section,
            'article': article_name, 'article_no': article.no,
            'article_branch_no': article.branch, 'title': title,
        }

        def path_meta(path, role):
            paragraph = next((n.no for n in path if n.level == 'paragraph'), None)
            item = next((n.no for n in path if n.level == 'item'), None)
            subitem = next((n.no for n in path if n.level == 'subitem'), None)
            return {
                **base, 'unit_level': path[-1].level, 'chunk_role': role,
                'search_enabled': role in {'child', 'anchor'},
                'paragraph': f'{paragraph}항' if paragraph else '',
                'paragraph_no': paragraph, 'item': f'{item}호' if item else '',
                'item_no': item, 'subitem': f'{subitem}목' if subitem else '',
                'subitem_no': subitem,
                'parent_id': parent_key, 'parent_point_id': parent_point_id,
            }

        def make_search_text(path, body):
            location = ' > '.join(filter(None, [law_name, chapter, section, title_label] + [
                location_segment(node) for node in path[1:]
            ]))
            ancestors = [node.own for node in path[:-1] if node.own]
            context = '\n'.join(ancestors[-2:])
            return '\n'.join(filter(None, [location, context, body]))

        if len(full_text) <= max_chars:
            key = f'{article_key}:child'
            add(full_text, make_search_text([article], full_text),
                {**path_meta([article], 'child'),
                 'parent_id': key, 'parent_point_id': stable_point_id(key)}, key)
            continue

        add(full_text, '', {
            **path_meta([article], 'parent'), 'search_enabled': False,
            'parent_id': None, 'parent_point_id': None,
        }, parent_key)

        def emit(path):
            node = path[-1]
            subtree = render(node)
            path_key = '/'.join(filter(None, [
                article_name, *[location_segment(n) for n in path[1:]]
            ]))
            if len(subtree) <= max_chars:
                key = f'{article_key}:{path_key}:child'
                add(subtree, make_search_text(path, subtree), path_meta(path, 'child'), key)
                return

            # 목록 때문에 긴 항·호의 도입 문장을 별도 대표 청크로 보존합니다.
            if node.level != 'article' and node.own:
                key = f'{article_key}:{path_key}:anchor'
                add(node.own, make_search_text(path, node.own),
                    path_meta(path, 'anchor'), key)

            if node.children:
                for child in node.children:
                    emit(path + [child])
                return

            for part_no, part in enumerate(split_leaf(subtree), start=1):
                key = f'{article_key}:{path_key}:child:{part_no}'
                add(part, make_search_text(path, part),
                    {**path_meta(path, 'child'), 'part': part_no}, key)

        for child in article.children:
            emit([article, child])

    # 부칙은 본문 조문과 섞지 않고 공포번호별 별도 검색 자료로 저장합니다.
    for addendum_index, addendum in enumerate(root.findall('./부칙/부칙단위'), start=1):
        raw = clean(addendum.findtext('./부칙내용', ''))
        if not raw:
            continue
        promulgation_no = direct_text(addendum, '부칙공포번호')
        for part_no, part in enumerate(split_leaf(raw), start=1):
            key = f'{LAW_VERSION}:{law_id}:addendum:{promulgation_no}:{addendum_index}:{part_no}'
            add(part, f'{law_name} 부칙 제{promulgation_no}호\n{part}', {
                'source_type': 'addendum', 'chapter': '', 'section': '',
                'article': '', 'article_no': None, 'article_branch_no': None,
                'title': '부칙', 'unit_level': 'addendum',
                'chunk_role': 'child', 'search_enabled': True,
                'paragraph': '', 'paragraph_no': None,
                'item': '', 'item_no': None, 'subitem': '', 'subitem_no': None,
                'parent_id': None, 'parent_point_id': None,
                'addendum_promulgation_no': promulgation_no,
                'addendum_promulgation_date': direct_text(addendum, '부칙공포일자'),
            }, key)

    # 제·개정이유도 법률 본문과 구분되는 별도 자료형으로 저장합니다.
    for reason_index, reason in enumerate(root.findall('./제개정이유/제개정이유내용'), start=1):
        raw = clean(''.join(reason.itertext()))
        for part_no, part in enumerate(split_leaf(raw), start=1):
            key = f'{LAW_VERSION}:{law_id}:reason:{reason_index}:{part_no}'
            add(part, f'{law_name} 제개정이유\n{part}', {
                'source_type': 'reason', 'chapter': '', 'section': '',
                'article': '', 'article_no': None, 'article_branch_no': None,
                'title': '제개정이유', 'unit_level': 'reason',
                'chunk_role': 'child', 'search_enabled': True,
                'paragraph': '', 'paragraph_no': None,
                'item': '', 'item_no': None, 'subitem': '', 'subitem_no': None,
                'parent_id': None, 'parent_point_id': None,
            }, key)

    return chunks

chunks = build_chunks(root)
print('전체 청크:', len(chunks))
print('역할:', Counter(c['chunk_role'] for c in chunks))
print('검색 단위:', Counter(c['unit_level'] for c in chunks if c['search_enabled']))
print('검색 대상:', sum(c['search_enabled'] for c in chunks))
assert all(len(c['content']) > 0 for c in chunks)
assert all(c['chunk_role'] != 'parent' or not c['search_enabled'] for c in chunks)

전체 청크: 118
역할: Counter({'child': 110, 'parent': 7, 'anchor': 1})
검색 단위: Counter({'paragraph': 44, 'article': 39, 'item': 12, 'subitem': 11, 'addendum': 3, 'reason': 2})
검색 대상: 111


## 2. 제2조 정의 청킹 확인

제2조제4호는 정의를 담은 `anchor`와 적용 영역을 담은 각 목으로 분리되어야 합니다.

In [3]:
article_2 = [c for c in chunks if c['article'] == '제2조']
print('제2조 전체 청크:', len(article_2))
for chunk in article_2:
    print(
        f"[{chunk['chunk_role']}/{chunk['unit_level']}]",
        chunk['article'], chunk['paragraph'], chunk['item'], chunk['subitem'],
        f"검색={chunk['search_enabled']} 길이={len(chunk['content'])}",
    )
    print(chunk['content'][:300], '\n')

high_impact_anchor = [
    c for c in article_2
    if c['chunk_role'] == 'anchor' and c['item_no'] == '4'
]
assert high_impact_anchor, '제2조제4호 고영향 인공지능 anchor가 없습니다.'
print('고영향 인공지능 anchor 확인 완료')

제2조 전체 청크: 24
[parent/article] 제2조    검색=False 길이=1926
제2조(정의) 이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2026.1.20>
1. "인공지능"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다.
2. "인공지능시스템"이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다.
3. "인공지능기술"이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다.
4. "고 

[child/item] 제2조  1호  검색=True 길이=71
1. "인공지능"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다. 

[child/item] 제2조  2호  검색=True 길이=109
2. "인공지능시스템"이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다. 

[child/item] 제2조  3호  검색=True 길이=63
3. "인공지능기술"이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다. 

[anchor/item] 제2조  4호  검색=True 길이=109
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다. 

[child/subitem] 제2조  4호 가목 검색=True 길이=28
가. 「에너지법」 제2조제1호에 따른 에너지의 공급 

[child/subitem] 제2조  4호 나목 검색=True 길이=33
나. 「먹는물관리법」 제3조제1호

## 3. 임베딩 및 Qdrant 저장

검색용 청크만 임베딩합니다. 부모 청크는 0 벡터로 같은 컬렉션에 저장하고, 검색할 때 `search_enabled=True` 필터를 적용합니다. 이 셀을 다시 실행하면 이 노트북 전용 컬렉션만 재생성합니다.

In [ ]:
embedding_client = get_embedding_model()
qdrant_client = get_qdrant_client()

search_chunks = [c for c in chunks if c['search_enabled']]

# 성공한 배치는 유지하고, 429가 발생한 현재 배치만 65초 후 재시도합니다.
EMBEDDING_BATCH_SIZE = 20
RETRY_WAIT_SECONDS = 65
search_vectors = []

for start in range(0, len(search_chunks), EMBEDDING_BATCH_SIZE):
    batch = search_chunks[start:start + EMBEDDING_BATCH_SIZE]
    batch_no = start // EMBEDDING_BATCH_SIZE + 1
    attempt = 0

    while True:
        attempt += 1
        try:
            batch_vectors = await embedding_client.aembed_documents(
                [chunk['search_text'] for chunk in batch]
            )
            search_vectors.extend(batch_vectors)
            print(
                f'임베딩 배치 {batch_no} 완료: '
                f'{len(search_vectors)}/{len(search_chunks)}개'
            )
            break
        except Exception as error:
            message = str(error).lower()
            is_rate_limit = (
                '429' in message
                or 'rate limit' in message
                or 'rate_limit' in message
            )
            if not is_rate_limit:
                raise

            print(
                f'배치 {batch_no} Rate Limit: '
                f'{RETRY_WAIT_SECONDS}초 후 재시도 '
                f'(현재 {attempt}회 실패)'
            )
            await asyncio.sleep(RETRY_WAIT_SECONDS)

vector_by_id = {c['point_id']: vector for c, vector in zip(search_chunks, search_vectors)}
vector_size = len(search_vectors[0])
zero_vector = [0.0] * vector_size

if qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.delete_collection(COLLECTION_NAME)

qdrant_client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
)

points = [
    PointStruct(
        id=chunk['point_id'],
        vector=vector_by_id.get(chunk['point_id'], zero_vector),
        payload={key: value for key, value in chunk.items() if key != 'search_text'},
    )
    for chunk in chunks
]
qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
print(f'{len(points)}개 저장 완료: {COLLECTION_NAME}')
print(f'검색 가능 {len(search_chunks)}개 / 부모 {len(points) - len(search_chunks)}개')

## 4. 검색 및 부모 조문 확장

검색 결과는 `child/anchor`만 나오며, 필요할 때 첫 결과의 전체 조문을 부모로 확장합니다.

In [ ]:
query = '고영향 인공지능이란 무엇인가요?'
query_vector = await embedding_client.aembed_query(query)

result = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    query_filter=Filter(must=[
        FieldCondition(key='search_enabled', match=MatchValue(value=True))
    ]),
    limit=5,
)

for rank, point in enumerate(result.points, start=1):
    p = point.payload
    print(
        rank, f"[{p['chunk_role']}/{p['unit_level']}]",
        p['article'], p['paragraph'], p['item'], p['subitem'],
        p['title'], f'score={point.score:.4f}',
    )
    print(p['content'][:800], '\n')

top_payload = result.points[0].payload
parent_point_id = top_payload.get('parent_point_id')
if parent_point_id and parent_point_id != result.points[0].id:
    parent = qdrant_client.retrieve(
        collection_name=COLLECTION_NAME,
        ids=[parent_point_id],
        with_payload=True,
    )
    if parent:
        print('\n[답변용 전체 부모 조문]')
        print(parent[0].payload['content'])
else:
    print('\n이 결과는 짧은 조문이므로 검색 청크 자체가 전체 조문입니다.')

## 평가 노트북 연결

평가 코드의 컬렉션 이름을 아래 값으로 변경하고, Dense 검색에도 같은 `search_enabled=True` 필터를 적용하세요.

```python
COLLECTION_NAME = 'ai_basic_law_adaptive_anchor_v1'
```

비교 대상은 기존 청킹 컬렉션과 이 컬렉션이며 `Hit@5`, `Recall@5`, `MRR@5`를 동일한 Golden Set으로 측정합니다.